# Lab 1a - Three Flavours of Machine Learning

**Presenter: Nagarjuna Naidu** | AI Solution Architect Program, Day 1 | ~15 minutes

**You will:**
- see supervised, unsupervised and reinforcement learning side by side
- change one parameter in each experiment, predict the effect first, then run it
- name the *feedback signal* each style needs (labels / none / rewards)

## 0. Environment check
**Why:** 90% of "it doesn't work on my machine" in a workshop is a version issue. We print versions once, fix the random seed, and set a consistent chart style so every plot looks the same on the projector.

**Architect lens:** Pinning versions and seeds is the smallest possible instance of *reproducibility*, which later becomes a governance requirement (audit: "can you reproduce the model that made this decision?").

In [ ]:
import os
# Keep BLAS/OpenMP to one thread: tiny data, so threading only adds overhead (and nondeterminism) on shared laptops/servers.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
import sys, platform, warnings, json, time, hashlib, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")          # keep the projector output clean

# ---- reproducibility: ONE seed used everywhere ----
SEED = 42
rng = np.random.default_rng(SEED)

# ---- consistent palette ----
INK, TEAL, LIME, CORAL, GOLD = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.labelcolor": INK,
    "text.color": INK, "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.prop_cycle": matplotlib.cycler(color=[TEAL, CORAL, GOLD, LIME, INK]),
    "axes.grid": True, "grid.alpha": 0.25,
})

# ---- where artefacts (saved models etc.) go: relative to the labs folder ----
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(f"Python      : {platform.python_version()}")
print(f"numpy       : {np.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"scipy       : {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print(f"Seed        : {SEED}   |   artefacts dir: {ARTIFACTS.resolve().name}/")
LAB_START = time.time()

# Part B - Three flavours of ML in 15 minutes
What separates the three families is not the algorithm, it is the **feedback signal**:

| Family | What the system is given | Feedback signal | Example question |
|---|---|---|---|
| Supervised | Inputs **and correct answers** (labels) | "You were right / wrong" per example | Will this customer churn? |
| Unsupervised | Inputs only | None - find structure yourself | What natural customer segments exist? |
| Reinforcement | An environment to act in | A **reward** after actions (often delayed) | Which sequence of actions maximises long-term reward? |

## B.1 Supervised learning - learning from labelled examples
**Why:** Imagine training a new loan officer by showing them 400 past applications *with the outcome*. They learn where the "approve / deny" line sits. That line is the **decision boundary**. We use just two features so you can literally *see* it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Synthetic loan applications: approve if income is high relative to debt, plus human-like noise.
n = 400
income_k = rng.uniform(20, 120, n)            # annual income, thousands
debt_ratio = rng.uniform(0.05, 0.9, n)         # debt-to-income
score = 0.06 * (income_k - 60) - 5.0 * (debt_ratio - 0.4) + rng.normal(0, 1.0, n)
approved = (score > 0).astype(int)
X_loan = np.column_stack([income_k, debt_ratio])

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_loan, approved, test_size=0.25, random_state=SEED, stratify=approved)
loan_model = LogisticRegression().fit(Xl_tr, yl_tr)

def plot_boundary(model, X, y, title, ax=None):
    """Colour the plane by what the model would predict, then overlay the real labelled points."""
    ax = ax or plt.gca()
    gx, gy = np.meshgrid(np.linspace(20, 120, 250), np.linspace(0.05, 0.9, 250))
    zz = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=["#F6C9BC", "#D9EDB0"], alpha=0.9)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c=CORAL, s=18, edgecolor="white", lw=0.4, label="denied")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c=TEAL, s=18, edgecolor="white", lw=0.4, label="approved")
    ax.set(xlabel="Income (k)", ylabel="Debt-to-income ratio", title=title)
    ax.legend(loc="upper right", fontsize=8)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_boundary(loan_model, Xl_te, yl_te, f"Logistic regression boundary (test acc = {loan_model.score(Xl_te, yl_te):.2f})", ax)
plt.show()
print("Learned weights  [income, debt_ratio]:", loan_model.coef_.round(3), "| intercept:", loan_model.intercept_.round(2))

**Your turn (3 min):** The straight line is *one* hypothesis. Train a `DecisionTreeClassifier(max_depth=...)` on the same data. Try `max_depth=2` and `max_depth=12`, and compare **train vs test accuracy**. Which one is memorising the training set (overfitting)?

In [ ]:
# TODO: fit a DecisionTreeClassifier with max_depth=12 on (Xl_tr, yl_tr) and print train & test accuracy.
tree = None
if tree is not None:
    print("train acc", tree.score(Xl_tr, yl_tr), "test acc", tree.score(Xl_te, yl_te))
else:
    print("Fill in the TODO above.")

**Architect takeaway (supervised):** needs **labelled historical outcomes** - the label *is* the expensive part (who produces it, how late does it arrive, is it biased?). The train-vs-test gap is your first overfitting alarm.

## B.2 Unsupervised learning - finding structure without answers
**Why:** Marketing asks "who are our customers?" - nobody has labelled them. K-Means groups points that are *close together*. The output is a hypothesis for humans to name ("bargain hunters", "high-value regulars"), **not** a truth.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Three hidden customer personas (we know them, the algorithm does not)
personas = [("Occasional", (25, 2), (8, 0.8), 150),
            ("Regular",    (70, 8), (12, 1.5), 200),
            ("Power user", (140, 18), (20, 2.5), 100)]
pts = []
for _, (m_spend, m_visit), (s_spend, s_visit), cnt in personas:
    pts.append(np.column_stack([rng.normal(m_spend, s_spend, cnt), rng.normal(m_visit, s_visit, cnt)]))
X_cust = np.vstack(pts)                            # columns: monthly spend, visits/month
X_cust_s = StandardScaler().fit_transform(X_cust)  # K-Means is distance-based -> put features on one scale

km = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X_cust_s)
print("silhouette (k=3):", round(silhouette_score(X_cust_s, km.labels_), 3), "(1.0 = perfectly separated, 0 = overlapping)")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for k, col in zip(range(3), [TEAL, CORAL, GOLD]):
    m = km.labels_ == k
    ax.scatter(X_cust[m, 0], X_cust[m, 1], s=16, c=col, alpha=0.8, label=f"segment {k} (n={m.sum()})")
centers = StandardScaler().fit(X_cust).inverse_transform(km.cluster_centers_)
ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=220, c=INK, edgecolor="white", label="centroids")
ax.set(xlabel="Monthly spend", ylabel="Visits per month", title="K-Means customer segments (unlabelled data)")
ax.legend(fontsize=8); plt.show()

**Your turn (3 min):** We told K-Means `k=3` because *we* built the data. In real life nobody tells you. Loop over `k = 2..6`, compute the silhouette score for each and pick the best `k`. Does the answer match the three personas we hid?

In [ ]:
# TODO: compute silhouette_score for k in range(2, 7) and store {k: score} in `sil`.
sil = {}
print("silhouette by k:", sil if sil else "Fill in the TODO above.")

**Architect takeaway (unsupervised):** needs **no labels** but needs a **human interpretation step** and a business definition of "useful". There is no accuracy to report - validate by *actionability* (do segments respond differently to offers?).

## B.3 Reinforcement learning - learning by trial, error and reward
**Why:** Think of training a dog. Nobody gives it a labelled dataset of "correct moves"; it tries things and gets a treat (reward) or not. Below, an agent learns to walk through a 5x5 grid from the top-left (S) to the goal (G, +10) while avoiding pits (-10), paying a small -0.1 for every step (so shorter is better).
The algorithm is **tabular Q-learning**: keep a table `Q[state, action]` = "how good is this move from here?" and nudge it after every step using the reward received.

```
Q[s,a] <- Q[s,a] + alpha * ( reward + gamma * max(Q[s',:]) - Q[s,a] )
```

In [ ]:
# --- A tiny grid world, numpy only ---
GRID = 5
START, GOAL, PITS = (0, 0), (4, 4), {(1, 3), (3, 1), (2, 2)}
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # up, down, left, right
ARROWS = ["^", "v", "<", ">"]

def step_env(state, a):
    r, c = state
    dr, dc = ACTIONS[a]
    nr, nc = min(max(r + dr, 0), GRID - 1), min(max(c + dc, 0), GRID - 1)   # walls: stay inside the grid
    nxt = (nr, nc)
    if nxt == GOAL:  return nxt, 10.0, True
    if nxt in PITS:  return nxt, -10.0, True
    return nxt, -0.1, False

def train_q(epsilon=0.2, episodes=600, alpha=0.5, gamma=0.95, seed=SEED, max_steps=60):
    """Epsilon-greedy Q-learning. epsilon = probability of EXPLORING (random move) instead of exploiting."""
    r_ = np.random.default_rng(seed)
    Q = np.zeros((GRID, GRID, 4))
    returns = []
    for ep in range(episodes):
        s, total = START, 0.0
        for _ in range(max_steps):
            a = r_.integers(4) if r_.random() < epsilon else int(np.argmax(Q[s]))   # explore vs exploit
            s2, rew, done = step_env(s, a)
            Q[s][a] += alpha * (rew + gamma * (0 if done else Q[s2].max()) - Q[s][a])   # the learning rule
            total += rew; s = s2
            if done: break
        returns.append(total)
    return Q, np.array(returns)

Q, returns = train_q(epsilon=0.2)
smooth = np.convolve(returns, np.ones(25) / 25, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(returns, color=TEAL, alpha=0.25, label="episode reward")
ax.plot(np.arange(len(smooth)) + 24, smooth, color=TEAL, lw=2.2, label="25-episode moving average")
ax.set(xlabel="Episode", ylabel="Total reward", title="Q-learning: reward improves with experience")
ax.legend(); plt.show()

print(f"Mean reward - first 50 episodes: {returns[:50].mean():6.2f}   last 50 episodes: {returns[-50:].mean():6.2f}")
print("\nLearned greedy policy (G=goal, X=pit; the agent starts top-left):")
for r in range(GRID):
    print("  " + " ".join("G" if (r, c) == GOAL else ("X" if (r, c) in PITS else ARROWS[int(np.argmax(Q[r, c]))]) for c in range(GRID)))

**Your turn (3 min):** RL has a famous dilemma - **explore vs exploit**. `epsilon` is the share of moves that are random. Train with `epsilon` in `[0.0, 0.05, 0.2, 0.5]` and compare the average reward **during the first 100 episodes** and **during the last 50**. Questions: (1) which setting earns the most *while learning*? (2) does a purely greedy agent (`epsilon=0`) always get stuck - or only sometimes?

In [ ]:
# TODO: for each epsilon, call train_q(epsilon=e) and fill results[e] = (mean reward first 100 episodes, mean reward last 50).
results = {}
for e in [0.0, 0.05, 0.2, 0.5]:
    pass
print(results if results else "Fill in the TODO above.")

**Architect takeaway (reinforcement):** needs a **simulator or safe environment** and a carefully designed **reward function** (a badly specified reward gets gamed!). Real-world uses: pricing, bidding, robotics, and - in a different form - RLHF for aligning LLMs. Data comes from *interaction*, so exploring in production has a real cost.

**Summary of Part B:** *Supervised = labelled answers. Unsupervised = structure only. Reinforcement = rewards from interaction.* In the next part we go deep on the family you will meet in 80% of enterprise work: supervised learning on tabular data, run through a proper lifecycle.

## Recap

| Style | Feedback signal | You must provide | Typical enterprise use |
|---|---|---|---|
| Supervised | Labels | Labelled examples | Fraud, churn, forecasting |
| Unsupervised | None | Raw data + human interpretation | Segmentation, anomaly detection |
| Reinforcement | Reward | An environment/simulator + a reward | Routing, pricing, RLHF for LLMs |

**Next:** tokens and embeddings, and Lab 2A tomorrow.